# 1. Аналитическое решение

Обозначим квадратичную функцию потерь:

$$
Q(\mathbf{X}, \mathbf{Y}, \mathbf{w}) = (\mathbf{Y} - \mathbf{Xw})^\top (\mathbf{Y} - \mathbf{Xw}) = \|\mathbf{Y} - \mathbf{Xw}\|_2^2
$$

где $\mathbf{X} = \begin{bmatrix} \mathbf{x}^{(1)}, \dots, \mathbf{x}^{(n)} \end{bmatrix}^\top$,
$\mathbf{x}^{(i)} \in \mathbb{R}^p$

Чтобы найти оптимальное решение, приравниваем градиент к нулю:

$$
\nabla_{\mathbf{w}} Q(\mathbf{w}) = \nabla_{\mathbf{w}} \left[ \mathbf{Y}^\top \mathbf{Y} - \mathbf{Y}^\top \mathbf{Xw} - \mathbf{w}^\top \mathbf{X}^\top \mathbf{Y} + \mathbf{w}^\top \mathbf{X}^\top \mathbf{Xw} \right] = 0
$$

Решая это уравнение, получаем:

$$
\hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{Y}
$$


### Вопросы на подумать:
> Радует что $\mathbf{X}^\top \mathbf{X}$ квадратная матрица, потому что иначе она не может быть обратимой (плохо обусловлена, сингулярная).

> Для неквадратных матриц может сущестовать: Псевдообратная матрица, Левый обратный элемент $A_L^{-1} A = I_n$, Правый обратный элемент $A A_R^{-1} = I_m$

> **что если эта матрица вырождена?**

> Почему $\mathbf{X}^\top \mathbf{X}$ — это ковариационная матрица?


In [51]:
n_features = 3
n_objects = 300
batch_size = 10
num_steps = 43
eps = 1e-3

w_true = np.random.normal(size=(n_features, ))

X = np.random.uniform(-5, 5, (n_objects, n_features))
X *= (np.arange(n_features) * 2 + 1)[np.newaxis, :]  # for different scales
X[:, -1] = X[:, -2] + np.random.uniform(-eps, eps, X[:, -2].shape)
Y = X.dot(w_true) + np.random.normal(0, 1, (n_objects))
w_0 = np.random.uniform(-2, 2, (n_features))

w_true

array([ 1.20777649, -1.08716727,  0.57508531])

* Если признаки сильно коррелированы (почти коллинеарны), то $X^\top X$ становится **почти сингулярной матрицей** (плохо обусловленной).


In [52]:
w_star = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(Y)
w_star

array([   1.20874946, -152.25205737,  151.735709  ])

* Обратим внимание:

  $$
  -152.25 + 151.74 \approx -0.51
  $$

То есть два "безумно больших" значения компенсируют друг друга и дают разумную сумму, близкую к истинному весу. Но само по себе такое решение **неустойчиво к шуму и плохо интерпретируемо**.


> Почему веса получаются большими?

В данной ситуации двум признакам достаточно информативности, чтобы хорошо аппроксимировать целевую переменную, а третий признак сильно коррелирует со вторым, возникает мультиколлинеарность. Это делает задачу оценки параметров плохо обусловленной: матрица $X^\top X$ становится почти вырожденной.

При этом модель стремится минимизировать эмпирическую ошибку, но из-за наличия шума в данных она начинает «подгоняться» под случайные флуктуации. Из-за высокой взаимной корреляции второго и третьего признаков, модель получает степень свободы, позволяющую компенсировать значения одного признака значениями другого. Чтобы "уловить" незначительные шумовые компоненты, веса при этих признаках становятся численно большими и противоположно направленными — иначе их вклад в модель был бы слишком мал для аппроксимации даже слабых флуктуаций в данных.


# 2. Регуляризация

**Невозможность обращения матрицы $\mathbf{X}^\top \mathbf{X}$**, если она:

* сингулярная (вырожденная), т.е. $\det(\mathbf{X}^\top \mathbf{X}) = 0$,
* или плохо обусловленная (почти вырождена, приводит к численной нестабильности при обращении).


## Решение: регуляризация Тихонова (ридж-регрессия)

Чтобы избежать сингулярности, применяют **регуляризацию Тихонова**, также известную как **ридж-регрессию**. Суть:

$$\large
\min_{\mathbf{w}} \left\| \mathbf{Y} - \mathbf{Xw} \right\|_2^2 + \lambda \left\| \mathbf{w} \right\|_2^2
$$

Где $\lambda > 0$ — коэффициент регуляризации.


## Аналитическое решение:

Регуляризованный функционал:

$$
L(\mathbf{w}) = (\mathbf{Y} - \mathbf{Xw})^\top (\mathbf{Y} - \mathbf{Xw}) + \lambda \mathbf{w}^\top \mathbf{w}
$$

Берём градиент и приравниваем к нулю:

$$
\nabla_{\mathbf{w}} L = -2\mathbf{X}^\top \mathbf{Y} + 2\mathbf{X}^\top \mathbf{X} \mathbf{w} + 2\lambda \mathbf{w} = 0
$$

Убираем множитель 2 и получаем:

$$
(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I}) \mathbf{w} = \mathbf{X}^\top \mathbf{Y}
$$


### Решение:

$$
\boxed{
\mathbf{w} = (\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{Y}
}
$$


### Почему теперь матрица обратима?

Добавление $\lambda \mathbf{I}$, где $\lambda > 0$, **смещает собственные значения** матрицы $\mathbf{X}^\top \mathbf{X}$ вверх и делает её **положительно определённой**, а значит — **обратимой**.


### Интуиция:

* Без регуляризации: модель может переобучаться и быть нестабильной из-за мультиколлинеарности.
* С регуляризацией: появляется контроль над сложностью модели и численная устойчивость.
